# Text Generation with an RNN (PyTorch)

A PyTorch port of the TensorFlow text generation tutorial. Trains a character-based GRU model on Shakespeare's writing from Andrej Karpathy's [The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/). Given a sequence of characters the model learns to predict the next character, enabling autoregressive text generation.

**Note:** Enable GPU acceleration to execute this notebook faster. In Colab: *Runtime > Change runtime type > Hardware accelerator > GPU*.

## Setup

### Import PyTorch and other libraries

In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
import time
import requests

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


### Download the Shakespeare dataset

Change the path below to use your own data.

In [2]:
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
path_to_file = 'shakespeare.txt'

if not os.path.exists(path_to_file):
    response = requests.get(url)
    with open(path_to_file, 'wb') as f:
        f.write(response.content)
    print(f'Downloaded {len(response.content):,} bytes to {path_to_file}')
else:
    print(f'File already exists: {path_to_file}')

Downloaded 1,115,394 bytes to shakespeare.txt


### Read the data

First, look at the text:

In [3]:
text = open(path_to_file, 'r', encoding='utf-8').read()
print(f'Length of text: {len(text)} characters')

Length of text: 1115394 characters


In [4]:
print(text[:250])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [6]:
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

65 unique characters


## Process the text

### Vectorize the text

Instead of `tf.keras.layers.StringLookup` we build simple Python dicts mapping characters ↔ integer IDs.

In [7]:
example_texts = ['abcdefg', 'xyz']

char2idx = {ch: idx for idx, ch in enumerate(vocab)}
idx2char = np.array(vocab)

for t in example_texts:
    ids = [char2idx[c] for c in t]
    print(f'{t!r} -> {ids}')

'abcdefg' -> [39, 40, 41, 42, 43, 44, 45]
'xyz' -> [62, 63, 64]


In [8]:
# Invert: IDs back to characters
for t in example_texts:
    ids = [char2idx[c] for c in t]
    recovered = ''.join(idx2char[ids])
    print(recovered)

abcdefg
xyz


In [9]:
def text_from_ids(ids):
    if isinstance(ids, torch.Tensor):
        ids = ids.cpu().numpy()
    return ''.join(idx2char[np.asarray(ids)])

### The prediction task

Given a sequence of characters, predict the next character at each time step.

### Create training examples and targets

Divide the text into non-overlapping chunks of `seq_length + 1` characters. Each chunk yields an `(input, target)` pair where `target` is `input` shifted one character to the right.

For example, with `seq_length = 4` and text `'Hello'`:
- input  → `'Hell'`
- target → `'ello'`

In [10]:
all_ids = np.array([char2idx[c] for c in text], dtype=np.int64)
all_ids

array([18, 47, 56, ..., 45,  8,  0])

In [11]:
for i in all_ids[:10]:
    print(idx2char[i])

F
i
r
s
t
 
C
i
t
i


In [12]:
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)

# Pack into non-overlapping chunks of seq_length+1
num_chunks = len(all_ids) // (seq_length + 1)
data = all_ids[:num_chunks * (seq_length + 1)].reshape(num_chunks, seq_length + 1)

for seq in data[:5]:
    print(repr(''.join(idx2char[seq])))

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '
'are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you k'
"now Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us ki"
"ll him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be d"
'one: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citi'


In [13]:
def split_input_target(sequence):
    input_text  = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

split_input_target(list('Tensorflow'))

(['T', 'e', 'n', 's', 'o', 'r', 'f', 'l', 'o'],
 ['e', 'n', 's', 'o', 'r', 'f', 'l', 'o', 'w'])

In [14]:
inp, tgt = split_input_target(data[0])
print('Input :', repr(text_from_ids(inp)))
print('Target:', repr(text_from_ids(tgt)))

Input : 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
Target: 'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '


### Create training batches

Use `torch.utils.data.Dataset` and `DataLoader` in place of `tf.data.Dataset`.

In [15]:
from torch.utils.data import Dataset, DataLoader


class ShakespeareDataset(Dataset):
    """Wraps the pre-chunked (N, seq_length+1) array."""
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        chunk = self.data[idx]
        return chunk[:-1], chunk[1:]


BATCH_SIZE = 64

dataset_obj = ShakespeareDataset(data)
dataloader  = DataLoader(
    dataset_obj,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=(device.type == 'cuda'),  # faster host->GPU transfers
)

print(f'Dataset : {len(dataset_obj)} sequences')
print(f'Batches : {len(dataloader)} x {BATCH_SIZE}')

for inp_batch, tgt_batch in dataloader:
    print(f'Input  shape: {inp_batch.shape}')
    print(f'Target shape: {tgt_batch.shape}')
    break

Dataset : 11043 sequences
Batches : 172 x 64
Input  shape: torch.Size([64, 100])
Target shape: torch.Size([64, 100])


## Build the Model

Three layers mirroring the Keras original:

| Keras | PyTorch |
|---|---|
| `Embedding` | `nn.Embedding` |
| `GRU` (return_sequences, return_state) | `nn.GRU(batch_first=True)` |
| `Dense` | `nn.Linear` |

In [16]:
vocab_size    = len(vocab)
embedding_dim = 256
rnn_units     = 1024

In [17]:
class MyModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, rnn_units, batch_first=True)
        self.dense = nn.Linear(rnn_units, vocab_size)

    def forward(self, x, states=None):
        x = self.embedding(x)            # (batch, seq, embed_dim)
        x, states = self.gru(x, states)  # (batch, seq, rnn_units)
        x = self.dense(x)                # (batch, seq, vocab_size)
        return x, states

    def get_initial_state(self, batch_size):
        dev = next(self.parameters()).device
        return torch.zeros(1, batch_size, self.gru.hidden_size, device=dev)

In [18]:
model = MyModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
).to(device)

print(model)

MyModel(
  (embedding): Embedding(65, 256)
  (gru): GRU(256, 1024, batch_first=True)
  (dense): Linear(in_features=1024, out_features=65, bias=True)
)


For each character the model looks up the embedding, runs the GRU one time-step, and applies the linear layer to produce logits over the vocabulary.

## Try the model

Check output shapes and sample from the untrained model.

In [19]:
for inp_batch, tgt_batch in dataloader:
    inp_batch = inp_batch.to(device)
    with torch.no_grad():
        example_batch_predictions, _ = model(inp_batch)
    print(example_batch_predictions.shape,
          '# (batch_size, sequence_length, vocab_size)')
    break

torch.Size([64, 100, 65]) # (batch_size, sequence_length, vocab_size)


In [20]:
# The model can run on inputs of any length
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}\n')
for name, param in model.named_parameters():
    print(f'  {name:30s} {str(list(param.shape)):25s} {param.numel():>10,} params')

Total trainable parameters: 4,021,569

  embedding.weight               [65, 256]                     16,640 params
  gru.weight_ih_l0               [3072, 256]                  786,432 params
  gru.weight_hh_l0               [3072, 1024]               3,145,728 params
  gru.bias_ih_l0                 [3072]                         3,072 params
  gru.bias_hh_l0                 [3072]                         3,072 params
  dense.weight                   [65, 1024]                    66,560 params
  dense.bias                     [65]                              65 params


In [21]:
# Sample from the output distribution (not argmax, to avoid loops)
probs = torch.softmax(example_batch_predictions[0], dim=-1)  # (seq, vocab)
sampled_ids = torch.multinomial(probs, num_samples=1).squeeze(-1)
sampled_ids = sampled_ids.cpu().numpy()

print('Input:\n', repr(text_from_ids(inp_batch[0].cpu())))
print()
print('Next Char Predictions:\n', repr(text_from_ids(sampled_ids)))

Input:
 "'s wish, not fierce and terrible\nOnly in strokes; but, with thy grim looks and\nThe thunder-like perc"

Next Char Predictions:
 "Aw3cUFbbJsYnmdBIFJe:fNteZP,3mEXLkp,v&w'k$HoLVfdUGO&Qm&ck:kFNT dViz&$&Lch,Ecje-V:Ya-rc. MovYb?ArmneP&"


## Train the model

### Attach an optimizer and a loss function

`nn.CrossEntropyLoss` is the PyTorch equivalent of `SparseCategoricalCrossentropy(from_logits=True)`. It expects raw logits and integer class indices.

In [22]:
loss_fn = nn.CrossEntropyLoss()

example_batch_mean_loss = loss_fn(
    example_batch_predictions.reshape(-1, vocab_size),
    tgt_batch.to(device).reshape(-1),
)
print(f'Prediction shape : {example_batch_predictions.shape}'
      ' # (batch_size, sequence_length, vocab_size)')
print(f'Mean loss        : {example_batch_mean_loss.item():.6f}')

Prediction shape : torch.Size([64, 100, 65]) # (batch_size, sequence_length, vocab_size)
Mean loss        : 4.183283


In [23]:
import math
print(f'exp(mean loss) = {math.exp(example_batch_mean_loss.item()):.5f}')
print(f'(should be close to vocab_size = {vocab_size})')

exp(mean loss) = 65.58082
(should be close to vocab_size = 65)


### Configure checkpoints

In [24]:
optimizer = torch.optim.Adam(model.parameters())

checkpoint_dir = './training_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

### Execute the training

In [25]:
EPOCHS = 20

In [26]:
history = []

for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0.0

    for batch_n, (inp, target) in enumerate(dataloader):
        inp    = inp.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        predictions, _ = model(inp)

        # predictions : (batch, seq_len, vocab_size)
        # CrossEntropyLoss needs  (N, C) and (N,)
        loss = loss_fn(predictions.reshape(-1, vocab_size),
                       target.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    elapsed  = time.time() - start
    history.append(avg_loss)
    print(f'Epoch {epoch+1:2d}/{EPOCHS}  Loss: {avg_loss:.4f}  '
          f'Time: {elapsed:.2f}s')

    if (epoch + 1) % 5 == 0:
        ckpt = os.path.join(checkpoint_dir, f'ckpt_epoch_{epoch+1}.pt')
        torch.save(model.state_dict(), ckpt)
        print(f'  -> Saved checkpoint: {ckpt}')

Epoch  1/20  Loss: 2.0109  Time: 8.89s
Epoch  2/20  Loss: 1.5167  Time: 8.69s
Epoch  3/20  Loss: 1.3929  Time: 8.73s
Epoch  4/20  Loss: 1.3258  Time: 8.72s
Epoch  5/20  Loss: 1.2775  Time: 8.77s
  -> Saved checkpoint: ./training_checkpoints/ckpt_epoch_5.pt
Epoch  6/20  Loss: 1.2360  Time: 9.01s
Epoch  7/20  Loss: 1.1979  Time: 9.17s
Epoch  8/20  Loss: 1.1600  Time: 9.44s
Epoch  9/20  Loss: 1.1233  Time: 9.79s
Epoch 10/20  Loss: 1.0862  Time: 9.81s
  -> Saved checkpoint: ./training_checkpoints/ckpt_epoch_10.pt
Epoch 11/20  Loss: 1.0488  Time: 9.69s
Epoch 12/20  Loss: 1.0096  Time: 9.77s
Epoch 13/20  Loss: 0.9720  Time: 9.80s
Epoch 14/20  Loss: 0.9343  Time: 9.85s
Epoch 15/20  Loss: 0.8970  Time: 9.96s
  -> Saved checkpoint: ./training_checkpoints/ckpt_epoch_15.pt
Epoch 16/20  Loss: 0.8629  Time: 10.40s
Epoch 17/20  Loss: 0.8283  Time: 10.44s
Epoch 18/20  Loss: 0.7976  Time: 10.43s
Epoch 19/20  Loss: 0.7686  Time: 10.31s
Epoch 20/20  Loss: 0.7439  Time: 10.17s
  -> Saved checkpoint: ./tr

## Generate text

Each call to `generate_one_step` feeds the current character(s) through the model and the GRU hidden state, then samples the next character from the output logits. The predicted character and updated state are passed back in on the next call.

The following `OneStep` class wraps the model for single-step generation:

In [27]:
class OneStep(nn.Module):
    def __init__(self, model, char2idx, idx2char, temperature=1.0):
        super().__init__()
        self.model       = model
        self.char2idx    = char2idx
        self.idx2char    = idx2char
        self.temperature = temperature
        self._device     = next(model.parameters()).device

    def generate_one_step(self, inputs, states=None):
        """
        inputs : list of strings, one per batch element.
                 On the first call pass the full prompt; on subsequent calls
                 pass the single predicted character.
        states : GRU hidden state (1, batch, rnn_units) or None.
        returns: list of predicted chars (one per batch), updated states.
        """
        # Convert strings -> token ID tensors
        input_ids = torch.tensor(
            [[self.char2idx.get(c, 0) for c in s] for s in inputs],
            dtype=torch.long,
            device=self._device,
        )

        with torch.no_grad():
            predicted_logits, states = self.model(input_ids, states)

        # Take only the last time-step prediction
        predicted_logits = predicted_logits[:, -1, :]        # (batch, vocab)
        predicted_logits = predicted_logits / self.temperature

        # Sample from the distribution
        probs         = torch.softmax(predicted_logits, dim=-1)
        predicted_ids = torch.multinomial(probs, num_samples=1).squeeze(-1)

        predicted_chars = [self.idx2char[i.item()] for i in predicted_ids]
        return predicted_chars, states

In [28]:
one_step_model = OneStep(model, char2idx, idx2char)

Run it in a loop to generate text. The model maintains its internal state across calls so it has full context.

In [29]:
start      = time.time()
states     = None
next_chars = ['ROMEO:']
result     = ['ROMEO:']

for n in range(1000):
    next_chars, states = one_step_model.generate_one_step(next_chars,
                                                           states=states)
    result.append(next_chars[0])

output_text = ''.join(result)
print(output_text, '\n\n' + '_' * 80)
print(f'\nRun time: {time.time() - start:.3f}s')

ROMEO:
So shall I live into the doors! know the comfort
Out--up disdainful king!

KING EDWARD IV:
They shall be stronger, that being contradiction,
To nothing can tempted retiment with nothing flatter.
Be king, not without your citizens
Under the voice of rushough shall be oldeed.
Here come the gone, the turns of you!

MAMILLIUS:
Dep your tongue!

AUFIDIUS:
O how the poor souls! What
vast thou a king of chatch'd by the way
and what you know this most comfort. But I be content.

QUEEN MARGARET:
My lord, we hear my brother wronged loan.
Where were is broil, thy life were acceptude,
He flatter'd him, ere I was in the vault,
I droved in thy life and all this garden,
To make an envious very sying.

FLORIZEL:
Shame he that came pardon or your
ears?

MENENIUS:
For that grief and defording them
thy charge.

BENVOLK:
I am as grave, as biggrange
Her witty prating hild, I'll refie
With this sea-storm? do me there?

BALTHASAR:
Why, how now, draw thy lady and points impatient for your turred accies

If you want the model to generate text *faster*, batch the generation. Below the model generates 5 outputs in roughly the same time as 1.

In [30]:
start      = time.time()
states     = None
next_chars = ['ROMEO:', 'ROMEO:', 'ROMEO:', 'ROMEO:', 'ROMEO:']
results    = [next_chars[:]]

for n in range(1000):
    next_chars, states = one_step_model.generate_one_step(next_chars,
                                                           states=states)
    results.append(next_chars[:])

# Reconstruct a full text string for each of the 5 sequences
num_seqs = len(results[0])
texts    = [''.join(step[j] for step in results) for j in range(num_seqs)]

for t in texts:
    print(t)
    print()
print(f'Run time: {time.time() - start:.3f}s')

ROMEO:
thou art the crown?

KING RICHARD II:
Ay, but I do feel our.

HENRY BOLINGBROKE:
I will go see it you to the wild-good clap-moubted countrymen,
That do you broother, she'll soonest
The proud threatening of Clifford often
Become the deputy bid him sleep.

VINCE EDWARD:
I had then be your tribunes!

COMINIUS:
I think most league it ordal, that beasts
From me here; call'st thou quiet lie.

First Murderer:
There hast thou know'st our cousin.

ISABELLA:
I think she's fortuned to't; but yet he
hath touch'd, when he did stand for him; but, as I lay me
From Richmond with curses! I may enter rain
and way to-morrow and gold and blood!
Stay, because that e'er wash 'twas received;
look yet I take my daughter, the king his power
Will entertain'd thy labour.

CAMILLO:
No, sir.

POLIXENES:
On edicious soldier.

GRUMIO:
I would you hear him bad?

DUCHESS OF YORK:
Then fly, I larmeth that I am here.
Nay, if you please to choose a case.
To prove, grave Perfumerle and the king
Had Barnardine and h

## Export the generator

Save and restore the model with `torch.save` / `torch.load`.

In [31]:
save_path = 'one_step_model.pt'
torch.save(model.state_dict(), save_path)
print(f'Model weights saved to {save_path}')

Model weights saved to one_step_model.pt


In [32]:
# Reload
reloaded_base = MyModel(vocab_size, embedding_dim, rnn_units).to(device)
reloaded_base.load_state_dict(
    torch.load(save_path, map_location=device, weights_only=True)
)
reloaded_base.eval()
one_step_reloaded = OneStep(reloaded_base, char2idx, idx2char)
print('Reloaded successfully')

Reloaded successfully


In [36]:
states     = None
next_chars = ['ROMEO:']
result     = ['ROMEO:']

for n in range(1000):
    next_chars, states = one_step_reloaded.generate_one_step(next_chars,
                                                              states=states)
    result.append(next_chars[0])

print(''.join(result))

ROMEO:
To me, I'll gave him my son Edward:
'Tis so vengeance on himself that act about the san
For that.

LUCENTIO:
Tell the process of thy prophecession, hath been
Decleding King Edward, and bristle victory!
My native baits, I can beat an eye
The mountain on the wings o' the Fown, revenge!
Now must I give him good to Catis,
When with her peaced brawling words in Wold cryshlect them to teach
The point of Clifford could the eyechery of the
duke, and break off good to his lecture
And he begot my stights as to tell the realm.

VOLUMNIA:
I prithee, sir; be Clarence: one's the deputed clay;
And so he that has left me from my centre,
To shake of all like an answer'd her to the cause.

WARWICK:
They have terror in her clocket here.
What say you? say, 'I'll grately serve, how from the
poor Henry's mischance of your voices and
Confess the wanton Edward, is froward, upon that rare
Were traitor or love, and thy best knees:
She could sweet matter than it is, he makes
Our brows as ever brought to t